# Registration testing

In [2]:
import json
import os
from pathlib import Path

import SimpleITK as sitk

In [3]:
os.getcwd()

'C:\\Users\\Wiktoria\\Documents\\GitHub\\robust-radiotherapy-planning\\notebooks'

In [4]:
# find project root (folder that contains .git)
ROOT = Path.cwd()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent

# set working directory to root
os.chdir(ROOT)

print("Now working in:", Path.cwd())

Now working in: C:\Users\Wiktoria\Documents\GitHub\robust-radiotherapy-planning


In [5]:
# Config
DATA_DICT = Path("src/data_full/data_dict.json")

SAVE_TRANSFORMS_DIR = Path("RESULTS/TRANSFORMS")
SAVE_TRANSFORMED_IMAGES_DIR = Path("RESULTS/TRANSFORMED_IMAGES")

SAVE_TRANSFORMS_DIR.mkdir(parents=True, exist_ok=True)
SAVE_TRANSFORMED_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
from ddf.registration.registration import multiscale_demons

In [ ]:
with open(DATA_DICT) as f:
    data_dict = json.load(f)

all_data = data_dict["0"]["train"] + data_dict["0"]["val"] + data_dict["test"]

if os.path.isfile("RESULTS/completed.json"):
    with open("RESULTS/completed.json") as f:
        completed = json.load(f)
else:
    completed = []

for item in all_data:
    fixedImName = item["fixed_image"]
    movingImName = item["moving_image"]

    patient_id = os.path.basename(fixedImName).split("_")[1]
    fixed_id = os.path.basename(fixedImName).split("_")[3]
    moving_id = os.path.basename(movingImName).split("_")[3]

    if [patient_id, fixed_id, moving_id] in completed:
        continue

    fixed = sitk.ReadImage(fixedImName, sitk.sitkFloat32)
    moving = sitk.ReadImage(movingImName, sitk.sitkFloat32)

    demons_filter = sitk.DiffeomorphicDemonsRegistrationFilter()  # type: ignore
    demons_filter.SetNumberOfIterations(120)  # type: ignore

    # Regularization (update field - viscous, total field - elastic)
    demons_filter.SetSmoothDisplacementField(True)  # type: ignore
    demons_filter.SetStandardDeviations(0.6)  # type: ignore

    # Create initial transform
    initial_transform = sitk.CenteredTransformInitializer(
        fixed,
        moving,
        sitk.Euler3DTransform(),  # type: ignore
        sitk.CenteredTransformInitializerFilter.GEOMETRY,
    )

    # Run the registration
    print(f"Starting registration: patient {patient_id}, fixed {fixed_id}, moving {moving_id}")
    try:
        tfm = multiscale_demons(
            registration_algorithm=demons_filter,
            fixed_image=fixed,
            moving_image=moving,
            initial_transform=initial_transform,
            shrink_factors=[16, 8, 4, 2],
            smoothing_sigmas=[16, 8, 4, 2],
        )
        print(f"Finished registration: patient {patient_id}, fixed {fixed_id}, moving {moving_id}")
    except Exception as e:
        print(f"Registration failed for patient {patient_id}, fixed {fixed_id}, moving {moving_id}: {e}")
        continue

    displacement_transform = sitk.DisplacementFieldTransform(tfm)  # type: ignore
    disp_field = sitk.TransformToDisplacementField(
        displacement_transform,
        sitk.sitkVectorFloat64,
        moving.GetSize(),  # type: ignore
        moving.GetOrigin(),  # type: ignore
        moving.GetSpacing(),  # type: ignore
        moving.GetDirection(),  # type: ignore
    )

    # Apply using ResampleImageFilter
    resampler = sitk.ResampleImageFilter()  # type: ignore
    resampler.SetReferenceImage(fixed)  # type: ignore
    resampler.SetInterpolator(sitk.sitkLinear)  # type: ignore
    resampler.SetDefaultPixelValue(0)  # type: ignore
    resampler.SetTransform(displacement_transform)  # type: ignore

    warped_image = resampler.Execute(moving)  # type: ignore

    fname = SAVE_TRANSFORMS_DIR / f"transform_patient_{patient_id}_fixed_{fixed_id}_moving_{moving_id}.nii.gz"
    sitk.WriteImage(disp_field, str(fname))

    fname = (
        SAVE_TRANSFORMED_IMAGES_DIR
        / f"transformed_image_patient_{patient_id}_fixed_{fixed_id}_moving_{moving_id}.nii.gz"
    )
    sitk.WriteImage(warped_image, str(fname))

    completed.append((patient_id, fixed_id, moving_id))

    with open("RESULTS/completed.json", "w") as f:
        json.dump(completed, f, indent=4)

Starting registration: patient 64, fixed 10, moving 1


# Using ready pipeline function

In [6]:
from ddf.registration.registration import run_demons_registration_pipeline

In [ ]:
run_demons_registration_pipeline(
    data_dict_path=DATA_DICT,
    save_transforms_dir=SAVE_TRANSFORMS_DIR,
    save_transformed_images_dir=SAVE_TRANSFORMED_IMAGES_DIR,
    completed_path=Path("RESULTS/completed.json"),
    iterations=5,
    shrink_factors=[4, 2],
    smoothing_sigmas=[4, 2],
)

Starting registration: patient 64, fixed 5, moving 1
Finished registration: patient 64, fixed 5, moving 1
Starting registration: patient 64, fixed 6, moving 1
Finished registration: patient 64, fixed 6, moving 1
